<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/SystemOneModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### The plan in plain words:

- Glue the question + the four boxes + the text into one input string.
- Run the model once (your prefill — the part before the decode loop).
- Look at the logits at the very last position (your logits[0, -1]).
- Out of 49,000 vocab logits, pull out just the four that correspond to "A", "B", "C", "D".
- Softmax those four. Done. Four probabilities, one per box.

No loop.

### Step 1 — Build the input

In [ ]:
state   = "Pikachu"
options = ["electric", "grass", "fire", "water"]

letters = ["A", "B", "C", "D"]
block   = "\n".join(f"{L}. {o}" for L, o in zip(letters, options))

prompt = f"""What type is this Pokemon?

{state}

A. electric
B. grass
C. fire
D. water

Answer:"""

**Difference from LLMs**. Earlier the prompt was "Once upon a time" — just the thing to continue. Here the prompt contains the question, the four boxes, and the thing to judge, all glued together. This is the "everything goes in as one input" idea, now as an actual string.

The prompt ends on "Answer:" deliberately. We're setting the model up so the very next token it would produce is a letter. We're not going to let it produce anything — we're going to peek at what it would have said.

### Step 2 — Tokenize

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tok   = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")
model.eval()

inputs = tok(prompt, return_tensors="pt")   # {'input_ids': (1, T), 'attention_mask': (1, T)}

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

### Step 3 — One forward pass

In [ ]:
with torch.no_grad():
    out = model(**inputs)
    logits = out.logits          # (1, T, vocab_size)

### Step 4 — The key step: grab only the box logits

In [ ]:
last_logits = logits[0, -1]      # (vocab_size,)  — the vote at the "Answer:" position

# token id for each letter (with a leading space, the way it appears after "Answer:")
option_ids = [tok.encode(f" {L}")[-1] for L in letters]   # e.g. [330, 396, 356, 413]

option_logits = last_logits[option_ids]    # (4,) — pull out just those four

**This is the entire trick.**

In _sample() function we did top-p: take the full vocab distribution, keep a slice, throw the rest away. This is the same move pushed to the extreme. Instead of keeping the nucleus, we keep exactly four tokens — the ones that spell your answer letters — and ignore the other ~49,000.

Line by line:

- `logits[0, -1]` — `logits[0, -1]` from `generate()`. The distribution at the final position.
- `tok.encode(" A")[-1]` — asks the tokenizer "what's the id for the token A?" It's one integer. We get four integers, one per letter.
- `last_logits[option_ids]` — fancy indexing. "From the 49,000 numbers, give me the four at these positions." Result: four raw scores.

### Step 5 — Softmax → one probability per box

In [ ]:
probs = torch.softmax(option_logits, dim=-1)     # (4,) sums to 1.0

for o, p in zip(options, probs.tolist()):
    print(f"{o:10s} {p:.3f}")

electric   0.249
grass      0.220
fire       0.281
water      0.249


**How the probability "spreads to each box" — the trace**

Here's the flow with illustrative numbers. Watch a single number turn into four:

```
                    prompt ends: "...D. water\n\nAnswer:"
                                      │
                        one forward pass through SmolLM
                                      │
                                      ▼
        last_logits  =  (49152 raw scores, one per vocab token)
                          ...
                          " A"  →  8.2      ← electric
                          " B"  →  4.1      ← grass
                          " C"  →  5.0      ← fire
                          " D"  →  4.7      ← water
                          " the"→ 9.9       ← model actually likes this most, we ignore it
                          ...
                                      │
             slice out ONLY the four letter positions
                                      │
                                      ▼
        option_logits = [8.2, 4.1, 5.0, 4.7]
                                      │
                                    softmax
                                      │  e^8.2=3641, e^4.1=60, e^5.0=148, e^4.7=110
                                      │  sum = 3959
                                      ▼
        probs = [3641/3959, 60/3959, 148/3959, 110/3959]
              = [0.920,     0.015,   0.037,    0.028]
                electric    grass    fire      water
```

## Full Code Block with **Qwen2.5-3B-Instruct**:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# The ONLY change from the previous script: a bigger, instruction-tuned model.
MODEL = "Qwen/Qwen2.5-3B-Instruct"      # try also: "meta-llama/Llama-3.2-3B-Instruct"

tok   = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto"
)
model.eval()


def classify(state, options, question="Pick the best matching label."):
    letters = [chr(ord("A") + i) for i in range(len(options))]
    block   = "\n".join(f"{L}. {o}" for L, o in zip(letters, options))

    # Instruction-tuned models expect the chat template, not a raw string.
    # (This is the one real addition vs the SmolLM version -- we wrap the
    #  prompt the way the model was fine-tuned to receive it.)
    user_msg = f"{question}\n\nText:\n{state}\n\nOptions:\n{block}\n\nReply with only the letter."
    prompt = tok.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )

    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        last_logits = model(**inputs).logits[0, -1]      # (vocab_size,)

    option_ids    = [tok.encode(f" {L}", add_special_tokens=False)[-1] for L in letters]
    option_logits = last_logits[option_ids]
    probs         = torch.softmax(option_logits.float(), dim=-1)
    return dict(zip(options, probs.tolist()))



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
if __name__ == "__main__":
    print("--- Pokemon typing ---")
    for o, p in classify("Pikachu", ["fire", "grass", "electric", "water"],
                         question="What type is this Pokemon?").items():
        print(f"  {o:10s} {p:.3f}")

    print("\n--- Support ticket routing ---")
    ticket = "Hi, we were billed twice for March. Please refund the duplicate."
    for o, p in classify(ticket, ["billing", "technical", "sales", "other"],
                         question="Which department should handle this?").items():
        print(f"  {o:10s} {p:.3f}")

    print("\n--- Urgency ---")
    for o, p in classify("The app is slow and I also want to upgrade my plan",
         ["billing", "technical", "sales", "other"],
         question="Which department?").items():
        print(f"  {o:10s} {p:.3f}")

--- Pokemon typing ---
  fire       0.000
  grass      0.000
  electric   1.000
  water      0.000

--- Support ticket routing ---
  billing    1.000
  technical  0.000
  sales      0.000
  other      0.000

--- Urgency ---
  billing    0.019
  technical  0.981
  sales      0.000
  other      0.000
